In [ ]:
!pip install -q langgraph

In [ ]:
import operator

from IPython.display import Image
from langchain_core.runnables import Runnable
from langgraph.graph import END, START, StateGraph
from pathlib import Path
from typing import Annotated, TypedDict


def display_graph(runnable: Runnable, output_png: Path):
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

In [ ]:
class TextProcessingState(TypedDict):
    raw_text: str
    clean_text: str
    word_count: int
    tokens: list[str]
    log: Annotated[list[str], operator.add]

In [ ]:
def normalize(state: TextProcessingState):
    cleaned = state["raw_text"].lower().strip()
    return { "clean_text": cleaned, "log": [f"normalized ({len(cleaned)} chars)"] }

def tokenize(state: TextProcessingState):
    tokens = [t for t in state["clean_text"].split() if t.isalpha()]
    return { "tokens": tokens, "log": [f"tokenized ({len(tokens)} tokens)"] }

def count_words(state: TextProcessingState) -> dict:
    return { "word_count": len(state["tokens"]), "log": [f"counted: {len(state['tokens'])} words"] }

def report(state: TextProcessingState) -> dict:
    print("===== REPORT =====")
    print(f"Original     : {state['raw_text']!r}")
    print(f"Normalized   : {state['clean_text']!r}")
    print(f"Word count   : {state['word_count']}")

    print("Log:")
    for line in state["log"]:
        print("- ", line)

    return { "log": ["reported"] }

In [ ]:
graph_builder = StateGraph(TextProcessingState)
graph_builder.add_node("normalize", normalize)
graph_builder.add_node("tokenize", tokenize)
graph_builder.add_node("count", count_words)
graph_builder.add_node("report", report)

graph_builder.add_edge(START, "normalize")
graph_builder.add_edge("normalize", "tokenize")
graph_builder.add_edge("tokenize", "count")
graph_builder.add_edge("count", "report")
graph_builder.add_edge("report", END)

graph = graph_builder.compile()

In [ ]:
display_graph(graph, Path("/content/graph.png"))

In [ ]:
final_state = graph.invoke(
    input={
        "raw_text": "Hello, world! Today we are talking about LangGraph and how to build multi-agent systems."
    }
)

In [ ]:
final_state